In [86]:
import json
import os
import glob
from rdflib import Graph
import owlrl

ALL_CONTRACT_IDS = {f"C{i}" for i in range(1, 11)}
SCENARIO_FOLDER = "../dataset/scenarios"
ONTOLOGY_PATH = "../dataset/ontologies"

def format_term(term):
    """Cleans RDF terms for readable console output."""
    if term is None: return "N/A"
    val = str(term.toPython())
    return val.split('#')[-1] if 'http' in val else val

def print_pretty_sparql(query_text):
    """Formats the SPARQL for debugging failing sections."""
    fmt = query_text.replace('\\n', '\n').replace('\n', ' ')
    keywords = ["SELECT", "WHERE", "FILTER", "OPTIONAL", "UNION"]
    for kw in keywords:
        fmt = fmt.replace(kw, f"\n{kw}")
    print("\n   [ DEBUG: SPARQL QUERY ]")
    print("   " + fmt.strip().replace('\n', '\n   '))
    print("   " + "-" * 30)

def print_section(label, expected_ids, found_ids, sparql_rows=None):
    """Prints status and mismatch details for a specific category."""
    is_match = set(expected_ids) == set(found_ids)
    icon = "✅" if is_match else "❌"
    
    print(f"\n   {icon} {label}:")
    
    if sparql_rows:
        if not sparql_rows:
            print("      (No rows returned)")
        else:
            for row in sparql_rows:
                print(f"      | {' | '.join(format_term(t) for t in row)}")
    else:
        print(f"      IDs: {sorted(list(found_ids)) if found_ids else 'None'}")

    if not is_match:
        missing = set(expected_ids) - set(found_ids)
        extra = set(found_ids) - set(expected_ids)
        if missing: print(f"      ⚠️  MISSING: {sorted(list(missing))}")
        if extra:   print(f"      ⚠️  EXTRA:   {sorted(list(extra))}")

def process_scenarios(scen_range):
    print("="*80)
    print("INITIALIZING: Loading Ontologies & Reasoning...")
    g = Graph()
    g.parse(os.path.join(ONTOLOGY_PATH, "life_insurance_tbox.ttl"), format="turtle")
    g.parse(os.path.join(ONTOLOGY_PATH, "life_insurance_abox.ttl"), format="turtle")
    owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)
    print("READY.\n")

    files = glob.glob(os.path.join(SCENARIO_FOLDER, "*.json"))
    scenarios = []
    for f_path in files:
        with open(f_path, 'r') as f:
            data = json.load(f)
            scenarios.extend(data if isinstance(data, list) else [data])
    scenarios.sort(key=lambda x: x['scenario_id'])

    for scen in scenarios:
        sid = scen['scenario_id']
        try:
            num = int(sid.split('-')[-1])
        except: continue
        
        if not (scen_range[0] <= num <= scen_range[1]): continue
        
        json_cov = [r['contract_id'] for r in scen['contract_responses'] if r['status'] == "COVERED"]
        json_den = [r['contract_id'] for r in scen['contract_responses'] if r['status'] == "DENIED"]
        json_na = [r['contract_id'] for r in scen['contract_responses'] if r['status'] == "NOT_APPLICABLE"]

        res_cov = list(g.query(scen['query_covered']))
        res_den = list(g.query(scen['query_denied']))

        found_cov = {str(row[0].toPython()) for row in res_cov if row[0] is not None}
        found_den = {str(row[0].toPython()) for row in res_den if row[0] is not None}
        
        found_na = ALL_CONTRACT_IDS - (found_cov | found_den)

        match_c = (found_cov == set(json_cov))
        match_d = (found_den == set(json_den))
        match_na = (found_na == set(json_na))

        status = "PASS" if (match_c and match_d and match_na) else "FAIL"
        print(f"\n[{status}] {sid}: {scen['scenario_description'][:75]}...")
        print("-" * 80)

        if not match_c: print_pretty_sparql(scen['query_covered'])
        print_section("COVERED", json_cov, found_cov, res_cov)

        if not match_d: print_pretty_sparql(scen['query_denied'])
        print_section("DENIED", json_den, found_den, res_den)

        print_section("NOT_APPLICABLE (Calculated)", json_na, found_na)
        
        print("\n" + "."*80)


In [87]:
process_scenarios((1, 58))

INITIALIZING: Loading Ontologies & Reasoning...
READY.


[PASS] SCEN-001: Insured dies from natural causes (heart attack) 5 years after policy issue....
--------------------------------------------------------------------------------

   ✅ COVERED:
      | C1 | Section 5.1, 9.3 | We will pay the death benefit to the beneficiary if: (a) The insured dies while this policy is in force, and (b) We receive proof of death acceptable to us, and (c) The death occurs before the policy expiry date. We will pay valid claims within 30 days of receiving all required documentation.
      | C2 | Section 8.1, 8.2 | Upon receipt of proof of death, we will pay the beneficiary: Face Amount + Paid-Up Additions - Policy Loans - Loan Interest. Payment will be made within 7 days of receiving complete claim documentation.
      | C3 | Article 5.1, 5.4 | OPTION A (Level Death Benefit) - Currently Selected. Death Benefit = Specified Amount. Death benefit is paid minus any outstanding policy loans.
      | C4 | 